# NFL Draft Semantic Pillar Scoring

**Pipeline overview:**

| Phase | Step | What it does |
|-------|------|--------------|
| 1 | Normalization | Hyphens/dashes → spaces so `pass-rush` = `pass rush` |
| 1 | Curated stitching | Known NFL compound terms → single underscore tokens |
| 1 | Domain stop words | Keep directional adjectives; remove scouting filler |
| 1 | PMI discovery | Data-driven bigram stitching via NLTK collocations |
| 1 | Lemmatization | WordNet lemmatizer on remaining tokens |
| 2 | TF-IDF | Vectorize cleaned strengths text |
| 2 | Archetype scoring | Cosine similarity to 4 semantic pillar archetypes |
| 2 | 0–100 scaling | Min-max normalize each pillar score for readability |

## 0. Setup

In [43]:
import re
import warnings
import numpy as np
import pandas as pd
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

for resource in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

# ── Controls ──────────────────────────────────────────────────────────────────
PMI_TOP_N     = 30    # number of auto-discovered bigrams to stitch
PMI_MIN_FREQ  = 5     # minimum corpus frequency for bigram candidates
SCORE_COLS    = ['score_athletic', 'score_technical', 'score_character', 'score_iq']
TOP_N_DISPLAY = 30     # players to show per pillar in summary

## 1. Load Data

Expects a DataFrame `df` with at minimum: `player_name`, `position`, `strengths`.
Replace the sample block below with your own data source.

In [44]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')
df = df[['player_name', 'Pos_Group', 'position', 'strengths']].dropna(subset=['strengths'])

---
## Phase 1 — NFL-Way Preprocessing

### Step 1 — Normalization & Curated Phrase Stitching

In [45]:
# ── Curated NFL compound terms ─────────────────────────────────────────────────
# Applied AFTER hyphen normalization so 'pass-rush' and 'pass rush' both match.
# Sorted longest-first to prevent partial matches (trigrams before bigrams).
_CURATED_RAW = {
    # Trigrams
    'change of direction':  'change_of_direction',
    'low pad level':        'low_pad_level',
    'run after catch':      'run_after_catch',
    'yards after contact':  'yards_after_contact',
    'yards after catch':    'yards_after_catch',
    'off the line':         'off_the_line',
    'off the ball':         'off_the_ball',
    'point of attack':      'point_of_attack',
    # Bigrams
    'pass rush':            'pass_rush',
    'pass rusher':          'pass_rusher',
    'pass protection':      'pass_protection',
    'pass coverage':        'pass_coverage',
    'pad level':            'pad_level',
    'press coverage':       'press_coverage',
    'man coverage':         'man_coverage',
    'zone coverage':        'zone_coverage',
    'ball skills':          'ball_skills',
    'ball hawk':            'ball_hawk',
    'ball carrier':         'ball_carrier',
    'body control':         'body_control',
    'contact balance':      'contact_balance',
    'closing speed':        'closing_speed',
    'lateral quickness':    'lateral_quickness',
    'quick twitch':         'quick_twitch',
    'high motor':           'high_motor',
    'first step':           'first_step',
    'get off':              'get_off',
    'hand fighting':        'hand_fighting',
    'hand strength':        'hand_strength',
    'block shedding':       'block_shedding',
    'anchor strength':      'anchor_strength',
    'route running':        'route_running',
    'run blocking':         'run_blocking',
    'open field':           'open_field',
    'red zone':             'red_zone',
    'second level':         'second_level',
    'hip flexibility':      'hip_flexibility',
    'soft hands':           'soft_hands',
    'heavy hands':          'heavy_hands',
    'strong hands':         'strong_hands',
    'short area':           'short_area',
    'three down':           'three_down',
    'top end':              'top_end',
    'two gap':              'two_gap',
    'one gap':              'one_gap',
    'snap count':           'snap_count',
}

CURATED_PHRASE_MAP = dict(
    sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)

print(f'Curated phrases defined: {len(CURATED_PHRASE_MAP)}')

Curated phrases defined: 46


### Step 2 — Domain Stop Words

In [46]:
# Words NLTK would remove that carry real meaning in scouting language
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}

# Generic scouting filler — appears in virtually every report regardless of player
CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'work', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside', 'ceiling',
}

_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS

print(f'Base NLTK stops : {len(_base)}')
print(f'Un-stopped       : {len(KEEP_WORDS & _base)}  → {sorted(KEEP_WORDS & _base)}')
print(f'Custom added     : {len(CUSTOM_STOPS)}')
print(f'Final stop list  : {len(NFL_STOPWORDS)}')

Base NLTK stops : 198
Un-stopped       : 8  → ['above', 'below', 'down', 'off', 'out', 'over', 'through', 'up']
Custom added     : 35
Final stop list  : 225


### Step 3 — Preprocessing Function (applies Steps 1–2 + lemmatization)

In [47]:
lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str, phrase_map: dict = CURATED_PHRASE_MAP,
                   extra_phrases: dict = None) -> str:
    """
    NFL-aware preprocessing pipeline.
    1. Lowercase
    2. Normalize hyphens/dashes → spaces  (MUST be before stitching)
    3. Stitch curated + auto-discovered phrases
    4. Regex clean (keep underscores)
    5. Tokenize → stop word filter → lemmatize → length filter
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    text = text.lower()

    # Step 2a: normalize hyphens and dashes before stitching
    text = re.sub(r'[-\u2013\u2014]', ' ', text)

    # Step 2b: stitch curated phrases
    for phrase, token in phrase_map.items():
        text = text.replace(phrase, token)

    # Step 2c: stitch any PMI-discovered phrases (added after discovery cell)
    if extra_phrases:
        for phrase, token in extra_phrases.items():
            text = text.replace(phrase, token)

    # Step 3: regex — keep letters, underscores, spaces
    text = re.sub(r'[^a-z_\s]', ' ', text)

    # Step 4: tokenize
    tokens = text.split()

    # Step 5: stop word filter (always keep stitched tokens)
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]

    # Step 6: lemmatize non-stitched tokens
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]

    # Step 7: drop short single tokens
    tokens = [t for t in tokens if len(t) > 1]

    return ' '.join(tokens)


# Initial pass (curated phrases only — PMI phrases added after Step 4)
df['strengths_clean_v1'] = df['strengths'].apply(nfl_preprocess)
print('Preprocessing complete (pre-PMI).')
print('\nSample output:')
for _, row in df.head(3).iterrows():
    print(f"  [{row['position']}] {row['player_name']}")
    print(f"    RAW  : {row['strengths'][:120]}")
    print(f"    CLEAN: {row['strengths_clean_v1'][:120]}")
    print()

Preprocessing complete (pre-PMI).

Sample output:
  [WR] Seyi Ajirotutu
    RAW  : Ajirotutu has ideal height and a muscular body.  Long-strider with very good straight-line speed.  Fluid athlete that si
    CLEAN: ajirotutu ideal height muscular body long strider good straight line speed fluid athlete sink hip gain separation out br

  [DE] Rahim Alem
    RAW  : Alem is a high-motor kid who plays to the whistle every snap.  He can be a disruptive pass rusher who sheds blockers wit
    CLEAN: alem high_motor kid play whistle every snap disruptive pass_rusher shed blocker good hand technique nose football heady 

  [DT] Charles Alexander
    RAW  : Tall interior defender with a thick build and a quick first step.  Has impressive strength to fight through double teams
    CLEAN: tall interior defender thick build quick first_step impressive strength fight through double team heavy_hands engage pus



### Step 4 — PMI Bigram Discovery

Discover high-PMI bigrams from the corpus that the curated list may have missed.
Review the printed table — the auto-discovered phrases are stitched in the final preprocessing pass.

In [48]:
# Build token lists from the v1-cleaned text (curated phrases already stitched)
_token_lists = [
    [t for t in text.split() if '_' not in t]   # only un-stitched tokens
    for text in df['strengths_clean_v1']
]

finder = BigramCollocationFinder.from_documents(_token_lists)
finder.apply_freq_filter(PMI_MIN_FREQ)
scored = finder.score_ngrams(BigramAssocMeasures.pmi)

_already = set(CURATED_PHRASE_MAP.keys())

auto_candidates = [
    (w1, w2, round(score, 3), finder.ngram_fd[(w1, w2)])
    for (w1, w2), score in scored
    if  w1 not in NFL_STOPWORDS and w2 not in NFL_STOPWORDS
    and w1.isalpha() and w2.isalpha()
    and len(w1) > 2 and len(w2) > 2
    and f'{w1} {w2}' not in _already
][:PMI_TOP_N]

pmi_df = pd.DataFrame(auto_candidates, columns=['word1', 'word2', 'pmi', 'freq'])
pmi_df['phrase'] = pmi_df['word1'] + ' ' + pmi_df['word2']
pmi_df['token']  = pmi_df['word1'] + '_' + pmi_df['word2']

print(f'Top {PMI_TOP_N} auto-discovered bigrams (PMI ≥ threshold, freq ≥ {PMI_MIN_FREQ}):')
print('These will be stitched in the final preprocessing pass.\n')
print(pmi_df[['phrase', 'token', 'pmi', 'freq']].to_string(index=False))

# Build the auto-phrase map from discovered bigrams
AUTO_PHRASE_MAP = dict(zip(pmi_df['phrase'], pmi_df['token']))
# Sort longest-first (in case of overlapping phrases)
AUTO_PHRASE_MAP = dict(sorted(AUTO_PHRASE_MAP.items(), key=lambda x: len(x[0]), reverse=True))

Top 30 auto-discovered bigrams (PMI ≥ threshold, freq ≥ 5):
These will be stitched in the final preprocessing pass.

               phrase                 token    pmi  freq
        freight train         freight_train 15.062     8
             rag doll              rag_doll 15.062     9
          blue collar           blue_collar 14.910     7
         calling card          calling_card 14.910     7
        signal caller         signal_caller 14.910     6
       deeper deepest        deeper_deepest 14.700     7
          wake forest           wake_forest 14.688     6
           notre dame            notre_dame 14.647    12
        battering ram         battering_ram 14.621     9
       internal clock        internal_clock 14.358     6
      seeking missile       seeking_missile 13.969    10
           grows root            grows_root 13.869     7
             peek boo              peek_boo 13.840     6
          phone booth           phone_booth 13.773    20
           stat sheet       

In [49]:
# Final preprocessing pass — curated + PMI-discovered phrases
df['strengths_clean'] = df['strengths'].apply(
    lambda t: nfl_preprocess(t, extra_phrases=AUTO_PHRASE_MAP)
)

print('Final preprocessing complete.')
print(f'\nVocabulary size: {len(set(t for text in df["strengths_clean"] for t in text.split())):,} unique tokens')

# Count stitched tokens
_all_tokens = [t for text in df['strengths_clean'] for t in text.split()]
_stitched = Counter(t for t in _all_tokens if '_' in t)
print(f'Stitched phrase tokens found: {len(_stitched)}')
if _stitched:
    print('  Top stitched tokens:')
    for tok, cnt in _stitched.most_common(10):
        print(f'    {tok}: {cnt}')

Final preprocessing complete.

Vocabulary size: 7,740 unique tokens
Stitched phrase tokens found: 99
  Top stitched tokens:
    body_control: 461
    second_level: 458
    point_of_attack: 355
    open_field: 351
    pass_protection: 344
    change_of_direction: 291
    short_area: 282
    ball_skills: 273
    man_coverage: 237
    pad_level: 226


---
## Phase 2 — Semantic Pillar Scoring

### Step 5 — TF-IDF Vectorization

In [50]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,              # keep rare terms (small corpus); raise to 2-3 for large datasets
    max_df=0.95,           # drop near-universal terms
    sublinear_tf=True,     # dampen high-frequency terms
    token_pattern=r'[a-z_]+',   # allow underscore (stitched tokens)
)

player_matrix = vectorizer.fit_transform(df['strengths_clean'])

print(f'Player matrix : {player_matrix.shape}  (players × features)')
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')

# Spot-check: confirm stitched tokens are in vocabulary
_sample_tokens = ['pass_rush', 'pad_level', 'change_of_direction', 'high_motor',
                  'body_control', 'press_coverage', 'route_running', 'anchor_strength']
print('\nStitched tokens in TF-IDF vocabulary:')
for tok in _sample_tokens:
    present = tok in vectorizer.vocabulary_
    print(f'  {tok:30s} {"✓" if present else "–"}')

Player matrix : (6679, 149361)  (players × features)
Vocabulary size: 149,361

Stitched tokens in TF-IDF vocabulary:
  pass_rush                      ✓
  pad_level                      ✓
  change_of_direction            ✓
  high_motor                     ✓
  body_control                   ✓
  press_coverage                 ✓
  route_running                  ✓
  anchor_strength                ✓


### Step 6 — Pillar Archetype Definition & Scoring

In [51]:
# ── 4 Semantic Pillar Archetypes ───────────────────────────────────────────────
# Each string is a dense description of the pillar in scout-like language.
# Using domain terms (including stitched tokens) maximises vocabulary overlap.
ARCHETYPES = {
    'score_athletic': (
        'elite athletic speed explosive burst acceleration lateral quickness '
        'body_control change_of_direction closing_speed explosive_burst '
        'twitch twitchy agile agility flexible flexibility fluid fluidity '
        'vertical jump leap raw physical powerful strength powerful frame '
        'long athletic get_off first_step separation quick_twitch top_end'
    ),
    'score_technical': (
        'technique mechanics footwork hand_fighting hand_strength anchor_strength '
        'pad_level low_pad_level leverage pass_protection run_blocking '
        'route_running block_shedding press_coverage zone_coverage man_coverage '
        'precise precision timing consistent fundamental sound execution '
        'hip_flexibility balance short_area lateral_quickness polish refined '
        'crisp sharp release contested'
    ),
    'score_character': (
        'high_motor motor effort compete relentless hustle competitive '
        'toughness tough grit passion energy aggressive pursuit intensity '
        'leadership character discipline accountable driven focused '
        'diligent committed hard worker never quit battles emotional '
        'physical bully mentality edge'
    ),
    'score_iq': (
        'football iq intelligence smart instinct awareness anticipation '
        'recognition diagnosis read scheme assignment mental processing '
        'pre snap post snap understanding vision cerebral mature prepared '
        'coachable quick processor off_the_ball point_of_attack '
        'chess high football intelligence'
    ),
}

# Transform archetypes using the SAME fitted vectorizer (no refit)
archetype_texts  = [ARCHETYPES[col] for col in SCORE_COLS]
archetype_matrix = vectorizer.transform(archetype_texts)

print('Archetype vectors built.')
for col, vec in zip(SCORE_COLS, archetype_matrix):
    n_nonzero = vec.nnz
    print(f'  {col:25s}  non-zero features: {n_nonzero}')

Archetype vectors built.
  score_athletic             non-zero features: 49
  score_technical            non-zero features: 38
  score_character            non-zero features: 38
  score_iq                   non-zero features: 43


In [52]:
# ── Cosine similarity: each player × each archetype ───────────────────────────
sim_matrix = cosine_similarity(player_matrix, archetype_matrix)   # shape: (n_players, 4)

scores_raw = pd.DataFrame(sim_matrix, columns=SCORE_COLS)

# ── Scale 0–100 per pillar (min-max within pillar) ────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 100))
scores_scaled = pd.DataFrame(
    scaler.fit_transform(scores_raw),
    columns=SCORE_COLS
).round(1)

# ── Attach to main DataFrame ──────────────────────────────────────────────────
result = pd.concat(
    [df[['player_name', 'Pos_Group', 'position']].reset_index(drop=True), scores_scaled],
    axis=1
)

print('Pillar scores (0–100 scaled):')
print(result.to_string(index=False))

Pillar scores (0–100 scaled):
             player_name Pos_Group position  score_athletic  score_technical  score_character  score_iq
          Seyi Ajirotutu        WR       WR            42.6              0.0              0.0       0.0
              Rahim Alem      EDGE       DE             0.0             12.4             19.5      63.1
       Charles Alexander        DT       DT            20.3              0.0              0.0      10.6
       Danario Alexander        WR       WR            17.4              0.0              7.0       4.8
              Nate Allen        DB       FS             0.0             18.0              7.9       9.3
            Tyson Alualu      EDGE       DE             0.0              7.8             17.6      20.5
          Jonathon Amaya        DB       FS            25.4              0.0             23.9      22.2
             Pat Angerer        LB       LB             9.0              0.0             17.3      30.4
           Javier Arenas        DB

---
## Output — Results & Pillar Summaries

In [53]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Final DataFrame ────────────────────────────────────────────────────────────
print('=' * 65)
print('FINAL PILLAR SCORES')
print('=' * 65)
display_cols = ['player_name', 'Pos_Group', 'position'] + SCORE_COLS
print(result[display_cols].sort_values('score_athletic', ascending=False).to_string(index=False))

FINAL PILLAR SCORES
             player_name Pos_Group position  score_athletic  score_technical  score_character  score_iq
            Daniel Hardy      EDGE       DE           100.0              0.0              0.0       7.4
         Brandon Denmark      EDGE      OLB            98.4              0.0              0.0       0.0
          Menelik Watson        OL       OT            92.5              0.0              6.2       0.0
              Nate Askew      EDGE      OLB            91.9              0.0              0.0       4.6
              Riq Woolen        DB       CB            90.9              0.0              0.0       0.0
         Randell Johnson      EDGE      OLB            90.1              0.0              0.0       0.0
           Charles Brown        OL       OL            88.4              9.7              0.0      24.6
          Jonathan Owens        DB        S            87.6              0.0              0.0       7.9
            Devin Harper        LB       LB 

In [54]:
# ── Top-5 per pillar ───────────────────────────────────────────────────────────
PILLAR_LABELS = {
    'score_athletic':  'Athletic Profile',
    'score_technical': 'Technical Skills',
    'score_character': 'Competitive Character',
    'score_iq':        'Football IQ',
}

print('TOP 5 PLAYERS PER PILLAR')
print('=' * 65)
for col in SCORE_COLS:
    top = result.nlargest(TOP_N_DISPLAY, col)[['player_name', 'position', col]]
    print(f'\n{PILLAR_LABELS[col].upper()}')
    print(top.to_string(index=False))

TOP 5 PLAYERS PER PILLAR

ATHLETIC PROFILE
        player_name position  score_athletic
       Daniel Hardy       DE           100.0
    Brandon Denmark      OLB            98.4
     Menelik Watson       OT            92.5
         Nate Askew      OLB            91.9
         Riq Woolen       CB            90.9
    Randell Johnson      OLB            90.1
      Charles Brown       OL            88.4
     Jonathan Owens        S            87.6
       Devin Harper       LB            86.4
      Rodney Thomas       DB            79.8
   Tyler Boatwright       CB            79.7
       Jordan Brown       CB            78.7
       Roquan Smith       LB            78.0
     Michael Onuoha      OLB            77.3
      Joaquin Davis       WR            76.6
        J'Mon Moore       WR            75.0
         Nick Scott        S            74.4
    Blessuan Austin       CB            74.3
      Tre Tomlinson       CB            73.8
Adolphus Washington       DT            73.2
    Stansly 

In [55]:
# 1. Define the score columns
score_columns = ['score_athletic', 'score_technical', 'score_character', 'score_iq']

# 2. Group by the 'Pos_Group' column and calculate the median
median_scores_by_group = result.groupby('Pos_Group')[score_columns].median().round(1)

# 3. Sort by Athleticism (or any other pillar) to see the ranking
median_scores_by_group = median_scores_by_group.sort_values(by='score_athletic', ascending=False)

print("Median Pillar Scores by Position Group:")
print(median_scores_by_group)

Median Pillar Scores by Position Group:
           score_athletic  score_technical  score_character  score_iq
Pos_Group                                                            
WR                   14.8              7.6              0.0       3.1
EDGE                 14.7              0.0              5.4       6.1
DT                   12.4              7.0              4.6       7.2
TE                   12.0              7.8              0.0       3.0
DB                   11.8              7.6              3.2       6.2
RB                   10.6              5.8              0.0       3.3
LB                    9.6              5.8              4.9       7.4
OL                    8.9              8.5              0.0       3.7
QB                    3.5              6.8              0.0       7.2
SPECIAL               2.9              0.0              0.0       0.0


In [56]:
# 1. Pool all text by Position Group
pooled_text = df.groupby('Pos_Group')['strengths'].apply(lambda x: ' '.join(x.astype(str))).reset_index()

# 2. Vectorize the pooled text
# (Make sure to use the same vectorizer used for your archetypes)
pooled_matrix = vectorizer.transform(pooled_text['strengths'])

# 3. Calculate similarity between the Groups and Archetypes
group_sim = cosine_similarity(pooled_matrix, archetype_matrix)

# 4. Create the summary table
group_scores = pd.DataFrame(group_sim, columns=SCORE_COLS, index=pooled_text['Pos_Group'])

# 5. Scale 0-100 across the table for comparison
group_scores_scaled = (group_scores / group_scores.max().max() * 100).round(1)

print("Positional Pillar Importance (Pooled Analysis):")
print(group_scores_scaled.sort_values(by='score_athletic', ascending=False))

Positional Pillar Importance (Pooled Analysis):
           score_athletic  score_technical  score_character  score_iq
Pos_Group                                                            
EDGE                 65.2             25.3             75.8      58.0
TE                   65.1             34.0             63.3      44.3
LB                   63.6             22.3            100.0      72.3
DT                   61.7             25.5             77.5      48.6
RB                   61.4             22.6             56.3      45.2
DB                   60.0             23.4             77.4      63.0
WR                   57.6             35.4             64.5      43.1
OL                   54.3             24.7             64.8      64.7
QB                   33.9             34.8             57.1      96.1
SPECIAL              28.1             25.6             19.1      21.2
